## Notebook23d

In this notebook, we will see how to build a model to detect bounding boxes in a collection of images of birds.

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds-py/refs/heads/main/funs.py

In [ ]:
import numpy as np
import polars as pl
from ultralytics import YOLO

from funs import *
from plotnine import *
from polars import col as c
theme_set(theme_minimal())

ub = "https://raw.githubusercontent.com/taylor-arnold/fds-py-nb/refs/heads/main/"

In [ ]:
birds = pl.read_parquet(ub + "data/birds10.parquet")
birds_bbox = pl.read_csv(ub + "data/birds_1000.csv")

### Data Format

So far in this course, our image tasks have involved assigning a single label to an entire image. For example, "this is a cardinal" or "this is a blue jay." Bounding box detection is a fundamentally different task: instead of classifying the whole image, the model must locate *where* in the image the object of interest appears and draw a tight rectangle around it. This means each training example needs four numbers (the coordinates of the box corners) in addition to a class label.

Our bird bounding box dataset contains exactly this information. Each row specifies an image filepath, a species label, and four coordinates defining the corners of a bounding box around the bird.

### Preparing the YOLO Dataset

We will use **YOLO** (You Only Look Once), one of the most widely used object detection architectures. YOLO expects training data in a very specific directory layout: images and label files organized into `train`, `val`, and `test` splits, with a YAML configuration file that describes the dataset. Rather than setting this up by hand, we use a helper function that takes our Polars DataFrame and reorganizes it into the format YOLO requires.

### Training the Model

YOLO models are typically not trained from scratch. Instead, we start from a **pre-trained checkpoint** (`yolo11n.pt`) that has already learned general-purpose visual features on a large dataset, and then fine-tune it on our bird data. This is the same transfer learning idea we've seen before — the early layers already know how to detect edges, textures, and shapes, so we only need to teach the final layers what a "cardinal" or "blue jay" bounding box looks like.

Training runs for 50 epochs over our dataset. Since this takes a while, we wrap it in a try/except block that first checks for previously saved weights. If the trained model file already exists, we load it directly; otherwise we train from scratch and save the result.

### Visualizing Ground Truth

Before looking at what the model predicts, let's visualize the ground truth bounding boxes — the hand-labeled rectangles that tell us where the bird actually is in each image. This gives us a baseline to compare against and helps us understand what the model is trying to learn.

### Comparing Predictions to Ground Truth

Now we run the trained model on the same images and overlay its predicted bounding boxes (in salmon) alongside the ground truth boxes (in olive). This side-by-side comparison lets us visually assess how well the model is doing. Ideally, the two rectangles should overlap closely — but you'll notice that they rarely match perfectly, and occasionally the model may detect multiple objects or miss the bird entirely.

### Generating Predictions on the Full Dataset

To evaluate the model quantitatively, we need predictions on every image — not just the nine we visualized. The code below runs the model across the full dataset and collects the top-confidence predicted bounding box for each image into a new DataFrame. We take only the highest-confidence box per image because our dataset has exactly one bird per image, so we're asking: "where does the model think the bird most likely is?"

As with our other notebooks, we cache the results to a Parquet file so that re-running the notebook doesn't require re-computing all the predictions.

### Intersection over Union (IoU)

How do we measure whether a predicted bounding box is "correct"? We can't require an exact match — even human annotators would draw slightly different boxes. Instead, we use **Intersection over Union (IoU)**, the standard metric for bounding box quality.

IoU is the ratio of the area where the predicted and ground truth boxes overlap (the intersection) to the total area covered by both boxes combined (the union). An IoU of 1.0 means perfect overlap; an IoU of 0.0 means the boxes don't overlap at all. The standard threshold for counting a prediction as "correct" is IoU ≥ 0.5, meaning the predicted box must overlap with at least half of the ground truth box's area.

The computation below calculates the intersection rectangle (by taking the max of the left edges and the min of the right edges), computes its area, and then divides by the union area.

### Precision, Recall, and F1

Finally, we compute the same classification metrics we've used throughout the course — precision, recall, and F1 — but adapted for bounding box detection. A prediction counts as a **true positive** if the model produced a box with IoU ≥ some threshold against the ground truth. A **false positive** means the model predicted a box but it didn't sufficiently overlap with the true location. A **false negative** means the model failed to locate the bird at all (or its box was too far off). We will write a function for this:

Now, let's compute the function for a large set of cut-off values.

Common cut-off values to care about include 0.5, 0.8, and 0.95:

We could also visualize the metrics as a function of the cutoff value.